# CAMS

## CAMS EAC4 Download Pipeline — Dhaka AQI Study 
**Period:** 2017–2022 (BST) | **Resolution:** 0.75° × 0.75° | **Frequency:** 3-hourly

- `CAMS_VAR_MAP` updated with correct short names (`go3`, `no2`, `co`, `so2`) confirmed from diagnostic
- **BST timezone fix**: downloads `2016-12-31` as a 1-day UTC buffer so that after
  shifting UTC→BST (+6h) the output starts exactly at `2017-01-01 00:00 BST`
- Buffer row (`2016-12-31`) is trimmed from final CSV — output is clean 2017–2022 BST
- Bbox `[24.00, 90.00, 23.25, 90.75]` kept as-is (returns 2×2 grid points, mean taken)

### Output columns
`timestamp_utc`, `timestamp_bst`, `pm2_5_ugm3`, `pm10_ugm3`, `no2_ugm3`,
`o3_ugm3`, `co_mgm3`, `so2_ugm3`, `dust_aod_550nm`

### Workflow
```
→ Install & configure        (run every session)
→ Define function             (run every session)
→ Buffer day (2016-12-31)     (run once)
→ CAMS 2017  → cams_2017.csv
→ CAMS 2018  → cams_2018.csv
→ CAMS 2019  → cams_2019.csv
→ CAMS 2020  → cams_2020.csv
→ CAMS 2021  → cams_2021.csv
→ CAMS 2022  → cams_2022.csv
→ Merge all  → cams_3hourly_2017_2022_bst.csv
```

In [ ]:
# Install & configure
# Run every session
 
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "cdsapi>=0.7.7", "xarray", "netcdf4", "pandas"], check=True)
 
import os, pathlib, glob
 
ADS_TOKEN = "YOUR_ADS_API_KEY" # Replace with your ADS API KEY
pathlib.Path(os.path.expanduser("~/.adsapirc")).write_text(
    f"url: https://ads.atmosphere.copernicus.eu/api\nkey: {ADS_TOKEN}\n")
 
OUT     = "/kaggle/working/output"
CAMS_NC = f"{OUT}/cams_nc"
CSV_DIR = f"{OUT}/csv"
for d in [OUT, CAMS_NC, CSV_DIR]:
    os.makedirs(d, exist_ok=True)
 
CAMS_BBOX  = [24.00, 90.00, 23.25, 90.75]
CAMS_TIMES = ["00:00","03:00","06:00","09:00","12:00","15:00","18:00","21:00"]
 
# ── Group A: surface aerosols — no pressure level needed ─────────────────────
# PM2.5/PM10 units: kg m-3  → µg m-3 : × 1e9  (NO air density — already concentration)
# Dust AOD: dimensionless                      → keep as-is
CAMS_VARS_AEROSOL = [
    "particulate_matter_2.5um",
    "particulate_matter_10um",
    "dust_aerosol_optical_depth_550nm",
]
 
# ── Group B: reactive gases — require pressure_level "1000" (surface proxy) ──
# Units: kg kg-1 (mass mixing ratio)
#   → µg m-3 : × air_density(1.225) × 1e9
#   CO only  : × air_density(1.225) × 1e6  (reported in mg m-3)
CAMS_VARS_GAS = [
    "nitrogen_dioxide",
    "ozone",
    "carbon_monoxide",
    "sulphur_dioxide",
]
CAMS_GAS_LEVEL = "1000"   # hPa — closest pressure level to surface
 
AIR_DENSITY = 1.225   # kg m-3 at standard conditions
 
# ── Unit conversion map ───────────────────────────────────────────────────────
# PM: kg m-3  → µg m-3  (× 1e9, no air density)
# Gas: kg kg-1 → µg m-3 (× air_density × 1e9)
# CO:  kg kg-1 → mg m-3 (× air_density × 1e6)
# AOD: dimensionless (× 1.0)
CAMS_VAR_MAP = {
    # Aerosols (confirmed short names from diagnostic)
    "pm2p5":    ("pm2_5_ugm3",    1e9),                  # kg m-3 → µg m-3
    "pm10":     ("pm10_ugm3",     1e9),                  # kg m-3 → µg m-3
    "duaod550": ("dust_aod_550nm", 1.0),                 # dimensionless
    # Gases (kg kg-1 → concentration)
    "no2":      ("no2_ugm3",      AIR_DENSITY * 1e9),
    "go3":      ("o3_ugm3",       AIR_DENSITY * 1e9),
    "co":       ("co_mgm3",       AIR_DENSITY * 1e6),
    "so2":      ("so2_ugm3",      AIR_DENSITY * 1e9),
    # Fallback names
    "pm2_5":    ("pm2_5_ugm3",    1e9),
    "o3":       ("o3_ugm3",       AIR_DENSITY * 1e9),
}
 
print("✅ CAMS setup complete")
print(f"   Bbox           : {CAMS_BBOX}")
print(f"   Aerosol vars   : {CAMS_VARS_AEROSOL}  (surface, no level)")
print(f"   Gas vars       : {CAMS_VARS_GAS}  (pressure_level={CAMS_GAS_LEVEL} hPa)")
 
nc_files  = sorted(glob.glob(f"{CAMS_NC}/*.nc"))
csv_files = sorted(glob.glob(f"{CSV_DIR}/cams_*.csv"))
print(f"\n   NC files  : {len(nc_files)}")
print(f"   CSVs      : {[os.path.basename(f) for f in csv_files]}")

In [ ]:
# Define functions
# Run every session
 
import cdsapi, os, time, calendar, glob
import xarray as xr
import pandas as pd
 
 
def _get_ads_client():
    return cdsapi.Client(
        url="https://ads.atmosphere.copernicus.eu/api",
        key=open(os.path.expanduser("~/.adsapirc")).read().split("key:")[-1].strip(),
        quiet=False, retry_max=5, timeout=5400,
    )
 
 
def _fetch_cams(ads_client, request_dict, nc_path):
    """
    Submit one ADS request and save to nc_path.
    request_dict: full parameter dict passed to ads.retrieve()
    Returns True on success, False on failure.
    """
    if os.path.exists(nc_path) and os.path.getsize(nc_path) > 10_000:
        print(f"  ⏭  {os.path.basename(nc_path)} already done "
              f"({os.path.getsize(nc_path)/1e6:.1f} MB)")
        return True
 
    print(f"  ⏳ {os.path.basename(nc_path)} ...")
    t0 = time.time()
    try:
        ads_client.retrieve("cams-global-reanalysis-eac4",
                            request_dict, nc_path)
        size_mb = os.path.getsize(nc_path) / 1e6
        if size_mb < 0.01:
            raise RuntimeError(f"Empty file ({size_mb:.4f} MB) — "
                               "check ADS Terms of Use acceptance")
        print(f"  ✅ {os.path.basename(nc_path)}  "
              f"({size_mb:.1f} MB, {(time.time()-t0)/60:.1f} min)")
        return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(nc_path)} FAILED: {e}")
        if os.path.exists(nc_path):
            os.remove(nc_path)
        return False
 
 
def _base_request(date_str):
    """Common ADS fields shared by aerosol and gas requests."""
    return {
        "date":   date_str,
        "time":   CAMS_TIMES,
        "area":   CAMS_BBOX,
        "format": "netcdf",
    }
 
 
def _nc_to_df(nc_files, drop_level=False):
    """
    Open NC files, optionally select surface pressure level,
    spatial-mean, apply unit conversions, return DataFrame.
    """
    ds = xr.open_mfdataset(nc_files, combine="by_coords", engine="netcdf4")
 
    # Gas files have a pressure_level dim — select 1000 hPa (surface)
    if drop_level:
        level_dim = None
        for d in ["level", "pressure_level", "plev"]:
            if d in ds.dims:
                level_dim = d
                break
        if level_dim:
            # Select 1000 hPa — closest to surface
            lev_vals = ds[level_dim].values
            idx = abs(lev_vals - 1000).argmin()
            ds  = ds.isel({level_dim: idx})
            print(f"     Level selected: {lev_vals[idx]} hPa")
 
    lat_dims = [d for d in ds.dims if "lat" in d.lower()]
    lon_dims = [d for d in ds.dims if "lon" in d.lower()]
    df = (ds.mean(dim=lat_dims + lon_dims)
            .to_dataframe()
            .reset_index())
    ds.close()
 
    time_col = "time" if "time" in df.columns else "valid_time"
    df = df.rename(columns={time_col: "timestamp_utc"})
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"]).dt.tz_localize(None)
 
    converted = []
    for src, (dst, factor) in CAMS_VAR_MAP.items():
        if src in df.columns and dst not in df.columns:
            df[dst] = df[src] * factor
            converted.append(f"{src}→{dst}")
    print(f"     Converted : {converted}")
 
    return df
 
 
def download_cams_year(year):
    """
    Download CAMS EAC4 for `year`:
      - Aerosol request (surface): PM2.5, PM10, dust AOD
      - Gas request (1000 hPa level): NO2, O3, CO, SO2
 
    Also downloads the UTC buffer day (Dec 31, prev year) for BST alignment.
    Merges aerosol + gas, shifts UTC→BST, trims to {year} BST, saves CSV.
    """
    print(f"\n{'='*60}")
    print(f"CAMS download — {year}  (aerosol + gas, BST buffer)")
    print(f"{'='*60}")
 
    ads    = _get_ads_client()
    failed = []
    prev   = year - 1
 
    def date_str(y, m):
        last = calendar.monthrange(y, m)[1]
        return f"{y}-{m:02d}-01/{y}-{m:02d}-{last:02d}"
 
    # ── Build task list: (date_str, aerosol_nc, gas_nc) ──────────────────────
    tasks = [(
        f"{prev}-12-31/{prev}-12-31",
        f"{CAMS_NC}/buffer_{prev}_12_31_aerosol.nc",
        f"{CAMS_NC}/buffer_{prev}_12_31_gas.nc",
    )]
    for month in range(1, 13):
        ds_str = date_str(year, month)
        tasks.append((
            ds_str,
            f"{CAMS_NC}/{year}_{month:02d}_aerosol.nc",
            f"{CAMS_NC}/{year}_{month:02d}_gas.nc",
        ))
 
    # ── Download ──────────────────────────────────────────────────────────────
    for ds_str, aerosol_nc, gas_nc in tasks:
        tag = os.path.basename(aerosol_nc).replace("_aerosol.nc", "")
        print(f"\n  [{tag}]")
 
        # Aerosol request — surface, no pressure level
        ok_a = _fetch_cams(ads, {
            **_base_request(ds_str),
            "variable": CAMS_VARS_AEROSOL,
        }, aerosol_nc)
 
        # Gas request — must specify pressure_level
        ok_g = _fetch_cams(ads, {
            **_base_request(ds_str),
            "variable":       CAMS_VARS_GAS,
            "pressure_level": CAMS_GAS_LEVEL,
        }, gas_nc)
 
        if not ok_a: failed.append(f"{tag}_aerosol")
        if not ok_g: failed.append(f"{tag}_gas")
 
    if failed:
        print(f"\n  ⚠️  Failed: {failed}")
        print("  Re-run this cell to retry failed chunks only.")
        return False
 
    # ── Load and merge ────────────────────────────────────────────────────────
    print(f"\n  📊 Loading and merging aerosol + gas ...")
 
    aerosol_files = (
        [f"{CAMS_NC}/buffer_{prev}_12_31_aerosol.nc"] +
        sorted(glob.glob(f"{CAMS_NC}/{year}_*_aerosol.nc"))
    )
    gas_files = (
        [f"{CAMS_NC}/buffer_{prev}_12_31_gas.nc"] +
        sorted(glob.glob(f"{CAMS_NC}/{year}_*_gas.nc"))
    )
 
    if len(aerosol_files) < 13 or len(gas_files) < 13:
        print(f"  ⚠️  aerosol:{len(aerosol_files)}/13  gas:{len(gas_files)}/13")
        return False
 
    print("  Aerosol files:")
    df_aerosol = _nc_to_df(aerosol_files, drop_level=False)
 
    print("  Gas files:")
    df_gas = _nc_to_df(gas_files, drop_level=True)
 
    # Keep only converted output columns for merge
    aerosol_cols = ["timestamp_utc"] + [
        dst for _, (dst, _) in CAMS_VAR_MAP.items()
        if dst in df_aerosol.columns
    ]
    gas_cols = ["timestamp_utc"] + [
        dst for _, (dst, _) in CAMS_VAR_MAP.items()
        if dst in df_gas.columns
    ]
    # Deduplicate
    aerosol_cols = list(dict.fromkeys(aerosol_cols))
    gas_cols     = list(dict.fromkeys(gas_cols))
 
    df = pd.merge(
        df_aerosol[aerosol_cols],
        df_gas[gas_cols],
        on="timestamp_utc",
        how="inner",
    )
 
    # ── UTC → BST (+6h) ───────────────────────────────────────────────────────
    df["timestamp_bst"] = df["timestamp_utc"] + pd.Timedelta(hours=6)
 
    # ── Trim to exactly {year} in BST ─────────────────────────────────────────
    bst_start = pd.Timestamp(f"{year}-01-01 00:00:00")
    bst_end   = pd.Timestamp(f"{year}-12-31 21:00:00")
    df = df[(df["timestamp_bst"] >= bst_start) &
            (df["timestamp_bst"] <= bst_end)].copy()
 
    # ── Final column order ────────────────────────────────────────────────────
    keep = [
        "timestamp_utc", "timestamp_bst",
        "pm2_5_ugm3", "pm10_ugm3",
        "no2_ugm3", "o3_ugm3", "co_mgm3", "so2_ugm3",
        "dust_aod_550nm",
    ]
    keep   = [c for c in keep if c in df.columns]
    df_out = df[keep].sort_values("timestamp_bst").reset_index(drop=True)
 
    # ── Warn if any expected column is missing or all-zero ────────────────────
    expected = ["pm2_5_ugm3","pm10_ugm3","no2_ugm3","o3_ugm3",
                "co_mgm3","so2_ugm3","dust_aod_550nm"]
    for col in expected:
        if col not in df_out.columns:
            print(f"     ⚠️  MISSING column: {col}")
        elif df_out[col].max() == 0:
            print(f"     ⚠️  {col} is all zeros")
 
    # ── Save ──────────────────────────────────────────────────────────────────
    csv_path = f"{CSV_DIR}/cams_{year}.csv"
    df_out.to_csv(csv_path, index=False)
    print(f"\n  ✅ Saved: cams_{year}.csv")
    print(f"     Rows      : {len(df_out):,}")
    print(f"     Columns   : {list(df_out.columns)}")
    print(f"     BST range : {df_out['timestamp_bst'].iloc[0]} → "
          f"{df_out['timestamp_bst'].iloc[-1]}")
    print(f"     Size      : {os.path.getsize(csv_path)/1e6:.1f} MB")
    return True
 
 
print("✅ Functions ready: _fetch_cams(), _nc_to_df(), download_cams_year()")

In [ ]:
# CAMS 2017 ───────────────────────────────────────────────────────
download_cams_year(2017)

In [ ]:
# CAMS 2018 ───────────────────────────────────────────────────────
download_cams_year(2018)

In [ ]:
# CAMS 2019 ───────────────────────────────────────────────────────
download_cams_year(2019)

In [ ]:
# CAMS 2020 ───────────────────────────────────────────────────────
download_cams_year(2020)

In [ ]:
# CAMS 2021 ───────────────────────────────────────────────────────
download_cams_year(2021)

In [ ]:
# CAMS 2022 ───────────────────────────────────────────────────────
download_cams_year(2022)

In [ ]:
# Merge yearly CSVs → `cams_3hourly_2017_2022_bst.csv

import pandas as pd, os, glob

CSV_DIR  = "/kaggle/input/datasets/aditybarua07/cams-data"
OUT_PATH = "/kaggle/working/output/csv/cams_3hourly_2017_2022_bst.csv"

csv_files = sorted(glob.glob(f"{CSV_DIR}/cams_20??.csv"))
print(f"Found {len(csv_files)} yearly CSVs:")
for f in csv_files:
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)")

missing = [y for y in range(2017, 2023)
           if f"{CSV_DIR}/cams_{y}.csv" not in csv_files]
if missing:
    print(f"\n⚠️  Missing years: {missing} — run their download cells first.")
else:
    dfs = []
    for f in csv_files:
        df = pd.read_csv(f, parse_dates=["timestamp_utc", "timestamp_bst"])
        print(f"  {os.path.basename(f)}: {len(df):,} rows  "
              f"{df['timestamp_bst'].iloc[0]} → {df['timestamp_bst'].iloc[-1]}")
        dfs.append(df)

    merged = (pd.concat(dfs, ignore_index=True)
                .sort_values("timestamp_bst")
                .reset_index(drop=True))

    before = len(merged)
    merged = merged.drop_duplicates(subset="timestamp_bst")
    if len(merged) < before:
        print(f"  ⚠️  Dropped {before - len(merged)} duplicate BST timestamps")

    merged.to_csv(OUT_PATH, index=False)
    print(f"\n✅ Saved: cams_3hourly_2017_2022_bst.csv")
    print(f"   Rows   : {len(merged):,}")
    print(f"   Cols   : {list(merged.columns)}")
    print(f"   Period : {merged['timestamp_bst'].min()} → "
          f"{merged['timestamp_bst'].max()}")
    print(f"   Size   : {os.path.getsize(OUT_PATH)/1e6:.1f} MB")

# ERA5

## ERA5-Land Download Pipeline — Dhaka AQI Study

- CDS returns a **ZIP file** — every download is unzipped automatically after saving
- Time dimension is **`valid_time`** not `time`
- `blh` and `tcc` are downloaded in a **separate request** per fortnight and merged
- Each month split into **2 fortnights** to stay under CDS size limit
- **BST fix**: downloads `2016-12-31` as UTC buffer so output starts at `2017-01-01 00:00 BST`
- Files saved to `/kaggle/working/output/` — persists as Kaggle notebook output
- Yearly CSV written immediately after each year completes

### Variable short names
| Short name | Variable | Unit |
|---|---|---|
| `t2m` | 2m temperature | K |
| `d2m` | 2m dewpoint temperature | K |
| `u10` | 10m U wind component | m/s |
| `v10` | 10m V wind component | m/s |
| `tp` | Total precipitation | m |
| `sp` | Surface pressure | Pa |
| `ssrd` | Surface solar radiation downwards | J/m² |
| `blh` | Boundary layer height | m |
| `tcc` | Total cloud cover | 0–1 |

### Output columns
`timestamp_utc`, `timestamp_bst`, `t2m_celsius`, `dewpoint_celsius`,
`relative_humidity_pct`, `wind_speed_ms`, `wind_u_ms`, `wind_v_ms`,
`precip_mm`, `surface_pressure_hpa`, `boundary_layer_height_m`,
`solar_radiation_wm2`, `total_cloud_cover_fraction`

### Workflow
```
→ Install & configure        (run every session)
→ Define functions            (run every session)
→ ERA5 2017  → era5_2017.csv
→ ERA5 2018  → era5_2018.csv
→ ERA5 2019  → era5_2019.csv
→ ERA5 2020  → era5_2020.csv
→ ERA5 2021  → era5_2021.csv
→ ERA5 2022  → era5_2022.csv
→ Merge all  → era5_hourly_2017_2022_bst.csv
```

In [ ]:
# Install & configure
# Run every session
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "cdsapi>=0.7.7", "xarray", "netcdf4", "scipy", "pandas"],
               check=True)
 
import os, pathlib, glob
 
# ── Paste your CDS token ─────────────────────────────────────────────────────
CDS_TOKEN = "YOUR_CDS_API_KEY" # Replace with your CDS API KEY
pathlib.Path(os.path.expanduser("~/.cdsapirc")).write_text(
    f"url: https://cds.climate.copernicus.eu/api\nkey: {CDS_TOKEN}\n")
 
# ── Output directories ───────────────────────────────────────────────────────
OUT      = "/kaggle/working/output"
ERA5_NC  = f"{OUT}/era5_nc"
CSV_DIR  = f"{OUT}/csv"
for d in [OUT, ERA5_NC, CSV_DIR]:
    os.makedirs(d, exist_ok=True)
 
# ── Shared config ────────────────────────────────────────────────────────────
ERA5_HOURS = [f"{h:02d}:00" for h in range(24)]
 
# ERA5-Land bbox — 0.1° grid, 3×3 points over Dhaka corridor
# Covers: Kuril (23.82°N, 90.42°E), Uttara (23.85°N, 90.37°E), Tongi (23.89°N, 90.40°E)
ERA5_BBOX     = [24.0,  90.3,  23.8,  90.5]   # [N, W, S, E]
 
# ERA5 single levels bbox — 0.25° grid, expanded to guarantee 2×3 grid points
# Grid snaps to multiples of 0.25°: 23.75/24.00/24.25 lat, 90.00/90.25/90.50 lon
ERA5_ATM_BBOX = [24.00, 90.25, 23.75, 90.50]   # [N, W, S, E]
 
# ERA5-Land variables (confirmed present in downloaded files)
ERA5_VARS_MAIN = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "total_precipitation",
    "surface_pressure",
    "surface_solar_radiation_downwards",
]
 
# BLH + TCC — NOT in ERA5-Land, must come from ERA5 single levels
ERA5_VARS_ATM = [
    "boundary_layer_height",
    "total_cloud_cover",
]
 
print("✅ ERA5 setup complete")
print(f"\n   ERA5-Land bbox      : {ERA5_BBOX}  (0.1° grid → 3×3 points)")
print(f"   ERA5 single-lev bbox: {ERA5_ATM_BBOX}  (0.25° grid → 2×3 points)")
print(f"\n   ERA5-Land vars      : {ERA5_VARS_MAIN}")
print(f"   ERA5 single-lev vars: {ERA5_VARS_ATM}")
 
nc_files  = sorted(glob.glob(f"{ERA5_NC}/*.nc"))
csv_files = sorted(glob.glob(f"{CSV_DIR}/era5_*.csv"))
print(f"\n   NC files : {len(nc_files)}")
print(f"   CSVs     : {[os.path.basename(f) for f in csv_files]}")

In [ ]:
# CELL 2 — Define functions
# Run every session

import cdsapi, os, time, math, glob, calendar, zipfile, gzip, shutil
import xarray as xr
import pandas as pd


# ── Unzip/extract helper — detects actual file format ────────────────────────
def _unzip_nc(zip_path, target_nc):
    """
    Detect actual file format and extract to target_nc.
    Handles: ZIP, gzip, or raw NetCDF (no compression).
    """
    with open(zip_path, "rb") as f:
        header = f.read(8)

    # ZIP: starts with PK\x03\x04
    if header[:4] == b"PK\x03\x04":
        with zipfile.ZipFile(zip_path, "r") as z:
            nc_in_zip = [n for n in z.namelist() if n.endswith(".nc")]
            if not nc_in_zip:
                raise ValueError(f"No .nc inside ZIP. Contents: {z.namelist()}")
            z.extract(nc_in_zip[0], path=os.path.dirname(target_nc))
        extracted = os.path.join(os.path.dirname(target_nc), nc_in_zip[0])
        if extracted != target_nc:
            shutil.move(extracted, target_nc)
        os.remove(zip_path)

    # GZIP: starts with \x1f\x8b
    elif header[:2] == b"\x1f\x8b":
        with gzip.open(zip_path, "rb") as gz_in:
            with open(target_nc, "wb") as nc_out:
                shutil.copyfileobj(gz_in, nc_out)
        os.remove(zip_path)

    # Raw NetCDF3: starts with CDF\x01 or CDF\x02
    # Raw NetCDF4/HDF5: starts with \x89HDF
    elif header[:3] == b"CDF" or header[:4] == b"\x89HDF":
        shutil.move(zip_path, target_nc)

    else:
        raise ValueError(
            f"Unknown file format. Header hex: {header.hex()}\n"
            f"First bytes: {header}"
        )


# ── Fetch ERA5-Land (main met vars) ──────────────────────────────────────────
def _fetch_era5land(client, request_params, target_nc):
    """Download from reanalysis-era5-land, handle any format, save to target_nc."""
    if os.path.exists(target_nc) and os.path.getsize(target_nc) > 100_000:
        print(f"    ⏭  {os.path.basename(target_nc)} already done "
              f"({os.path.getsize(target_nc)/1e6:.2f} MB)")
        return True
    tmp_path = target_nc.replace(".nc", "_tmp.download")
    t0 = time.time()
    try:
        client.retrieve("reanalysis-era5-land", request_params, tmp_path)
        _unzip_nc(tmp_path, target_nc)
        print(f"    ✅ {os.path.basename(target_nc)}  "
              f"({os.path.getsize(target_nc)/1e6:.2f} MB, "
              f"{(time.time()-t0)/60:.1f} min)")
        return True
    except Exception as e:
        print(f"    ❌ ERA5-Land FAILED: {e}")
        for f in [tmp_path, target_nc]:
            if os.path.exists(f): os.remove(f)
        return False


# ── Fetch ERA5 single levels (BLH + TCC) ─────────────────────────────────────
def _fetch_era5sl(client, request_params, target_nc):
    """Download from reanalysis-era5-single-levels, handle any format, save to target_nc."""
    if os.path.exists(target_nc) and os.path.getsize(target_nc) > 10_000:
        print(f"    ⏭  {os.path.basename(target_nc)} already done "
              f"({os.path.getsize(target_nc)/1e6:.2f} MB)")
        return True
    tmp_path = target_nc.replace(".nc", "_tmp.download")
    t0 = time.time()
    try:
        client.retrieve("reanalysis-era5-single-levels", request_params, tmp_path)

        # Print header for debugging — remove once confirmed working
        with open(tmp_path, "rb") as f:
            header = f.read(8)
        print(f"    Format header: {header.hex()}  ({header})")

        _unzip_nc(tmp_path, target_nc)
        print(f"    ✅ {os.path.basename(target_nc)}  "
              f"({os.path.getsize(target_nc)/1e6:.2f} MB, "
              f"{(time.time()-t0)/60:.1f} min)")
        return True
    except Exception as e:
        print(f"    ❌ ERA5-SL FAILED: {e}")
        for f in [tmp_path, target_nc]:
            if os.path.exists(f): os.remove(f)
        return False


# ── Convert NC files → DataFrame ─────────────────────────────────────────────
def _nc_to_df(nc_files):
    """Open NC files, spatial-mean, return clean DataFrame with timestamp_utc."""
    ds = xr.open_mfdataset(nc_files, combine="by_coords", engine="netcdf4")
    lat_dims = [d for d in ds.sizes if "lat" in d.lower()]
    lon_dims = [d for d in ds.sizes if "lon" in d.lower()]
    df = (ds.mean(dim=lat_dims + lon_dims)
            .to_dataframe()
            .reset_index())
    ds.close()
    time_col = "valid_time" if "valid_time" in df.columns else "time"
    df = df.rename(columns={time_col: "timestamp_utc"})
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"]).dt.tz_localize(None)
    return df


# ── Main download function ────────────────────────────────────────────────────
def download_era5_year(year):
    """
    Downloads ERA5-Land (main met vars, 0.1°) and ERA5 single levels
    (BLH + TCC, 0.25°) for `year` plus a UTC buffer day (Dec 31 prev year).
    Merges both, converts UTC → BST (+6h),
    trims to exactly {year}-01-01 00:00 BST → {year}-12-31 23:00 BST.
    Writes era5_{year}.csv to CSV_DIR.
    """
    print(f"\n{'='*55}")
    print(f"ERA5 download — {year}")
    print(f"  ERA5-Land bbox : {ERA5_BBOX}")
    print(f"  ERA5-SL   bbox : {ERA5_ATM_BBOX}")
    print(f"{'='*55}")

    cds = cdsapi.Client(
        url="https://cds.climate.copernicus.eu/api",
        key=open(os.path.expanduser("~/.cdsapirc")).read().split("key:")[-1].strip(),
        quiet=False, retry_max=5, timeout=5400,
    )

    failed = []

    # ── Build chunk list ─────────────────────────────────────────────────────
    # Buffer day (Dec 31 prev year) + 12 months × 2 fortnights = 25 chunks
    prev   = year - 1
    chunks = [(f"buffer_{prev}_12_31", prev, "12", ["31"])]
    for month in range(1, 13):
        last_day = calendar.monthrange(year, month)[1]
        chunks.append((f"{year}_{month:02d}_A", year, f"{month:02d}",
                       [f"{d:02d}" for d in range(1,  16)]))
        chunks.append((f"{year}_{month:02d}_B", year, f"{month:02d}",
                       [f"{d:02d}" for d in range(16, last_day + 1)]))

    # ── Download each chunk ──────────────────────────────────────────────────
    for label, yr, mon, days in chunks:
        print(f"\n  Chunk: {label}")

        # ERA5-Land: main met vars, uses ERA5_BBOX (0.1°)
        ok = _fetch_era5land(cds, {
            "variable": ERA5_VARS_MAIN,
            "year":     [str(yr)],
            "month":    [mon],
            "day":      days,
            "time":     ERA5_HOURS,
            "area":     ERA5_BBOX,
            "format":   "netcdf",
        }, f"{ERA5_NC}/{label}_main.nc")
        if not ok: failed.append(f"{label}_main")

        # ERA5 single levels: BLH + TCC, uses ERA5_ATM_BBOX (0.25°)
        ok = _fetch_era5sl(cds, {
            "variable":     ERA5_VARS_ATM,
            "product_type": "reanalysis",
            "year":         [str(yr)],
            "month":        [mon],
            "day":          days,
            "time":         ERA5_HOURS,
            "area":         ERA5_ATM_BBOX,
            "format":       "netcdf",
        }, f"{ERA5_NC}/{label}_atm.nc")
        if not ok: failed.append(f"{label}_atm")

    if failed:
        print(f"\n  ⚠️  Failed chunks: {failed} — re-run this cell to retry")
        return False

    # ── Load all NC files ────────────────────────────────────────────────────
    main_files = sorted(
        glob.glob(f"{ERA5_NC}/buffer_{year-1}_12_31_main.nc") +
        glob.glob(f"{ERA5_NC}/{year}_*_main.nc"))
    atm_files  = sorted(
        glob.glob(f"{ERA5_NC}/buffer_{year-1}_12_31_atm.nc") +
        glob.glob(f"{ERA5_NC}/{year}_*_atm.nc"))

    expected = 25
    if len(main_files) < expected or len(atm_files) < expected:
        print(f"  ⚠️  Incomplete: {len(main_files)} main, "
              f"{len(atm_files)} atm (expected {expected}) — skipping CSV")
        return False

    print(f"\n  📊 Converting to CSV ...")
    df_main = _nc_to_df(main_files)
    df_atm  = _nc_to_df(atm_files)

    # Merge BLH + TCC onto main DataFrame
    atm_keep = ["timestamp_utc"] + [c for c in ["blh", "tcc"]
                                     if c in df_atm.columns]
    df = df_main.merge(df_atm[atm_keep], on="timestamp_utc", how="left")
    print(f"     Merged columns : {list(df.columns)}")

    # ── Derived variables ────────────────────────────────────────────────────
    if "t2m" in df.columns:
        df["t2m_celsius"]      = df["t2m"]  - 273.15
    if "d2m" in df.columns:
        df["dewpoint_celsius"] = df["d2m"]  - 273.15

    if "t2m_celsius" in df.columns and "dewpoint_celsius" in df.columns:
        T, Td = df["t2m_celsius"], df["dewpoint_celsius"]
        df["relative_humidity_pct"] = (100 * (
            (17.625 * Td / (243.04 + Td)).apply(math.exp) /
            (17.625 * T  / (243.04 + T )).apply(math.exp)
        )).clip(0, 100)

    if "u10" in df.columns and "v10" in df.columns:
        df["wind_speed_ms"] = (df["u10"]**2 + df["v10"]**2)**0.5
        df = df.rename(columns={"u10": "wind_u_ms", "v10": "wind_v_ms"})

    if "tp"   in df.columns:
        df["precip_mm"]            = df["tp"]   * 1000
    if "sp"   in df.columns:
        df["surface_pressure_hpa"] = df["sp"]   / 100
    if "ssrd" in df.columns:
        df["solar_radiation_wm2"]  = df["ssrd"] / 3600
    if "blh"  in df.columns:
        df = df.rename(columns={"blh": "boundary_layer_height_m"})
    if "tcc"  in df.columns:
        df = df.rename(columns={"tcc": "total_cloud_cover_fraction"})

    # ── UTC → BST (+6h) ──────────────────────────────────────────────────────
    df["timestamp_bst"] = df["timestamp_utc"] + pd.Timedelta(hours=6)

    # ── Trim to exactly {year} in BST ────────────────────────────────────────
    df = df[
        (df["timestamp_bst"] >= pd.Timestamp(f"{year}-01-01 00:00:00")) &
        (df["timestamp_bst"] <= pd.Timestamp(f"{year}-12-31 23:00:00"))
    ].copy()

    # ── Select and order final columns ────────────────────────────────────────
    keep = [
        "timestamp_utc", "timestamp_bst",
        "t2m_celsius", "dewpoint_celsius", "relative_humidity_pct",
        "wind_speed_ms", "wind_u_ms", "wind_v_ms",
        "precip_mm", "surface_pressure_hpa",
        "boundary_layer_height_m", "solar_radiation_wm2",
        "total_cloud_cover_fraction",
    ]
    keep   = [c for c in keep if c in df.columns]
    df_out = df[keep].sort_values("timestamp_bst").reset_index(drop=True)

    # ── Save ──────────────────────────────────────────────────────────────────
    csv_path = f"{CSV_DIR}/era5_{year}.csv"
    df_out.to_csv(csv_path, index=False)

    print(f"\n  ✅ Saved: era5_{year}.csv")
    print(f"     Rows       : {len(df_out):,}")
    print(f"     Columns    : {list(df_out.columns)}")
    print(f"     BST range  : {df_out['timestamp_bst'].iloc[0]} → "
          f"{df_out['timestamp_bst'].iloc[-1]}")
    print(f"     Size       : {os.path.getsize(csv_path)/1e6:.2f} MB")

    for col in ["boundary_layer_height_m", "total_cloud_cover_fraction"]:
        if col not in df_out.columns:
            print(f"     ⚠️  {col} missing — check ERA5-SL download")

    return True


print("✅ All functions defined")
print("   _unzip_nc()       — detects ZIP / gzip / raw NetCDF and extracts")
print("   _fetch_era5land() — downloads ERA5-Land vars (0.1°)")
print("   _fetch_era5sl()   — downloads ERA5 single-level vars (0.25°)")
print("   _nc_to_df()       — converts NC files to DataFrame")
print("   download_era5_year(year) — runs full pipeline for one year")

In [ ]:
# ERA5 2017 ────────────────────────────────────────────────────────
download_era5_year(2017)

In [ ]:
# ERA5 2018 ────────────────────────────────────────────────────────
download_era5_year(2018)

In [ ]:
# ERA5 2019 ────────────────────────────────────────────────────────
download_era5_year(2019)

In [ ]:
# ERA5 2020 ────────────────────────────────────────────────────────
download_era5_year(2020)

In [ ]:
# ERA5 2021 ────────────────────────────────────────────────────────
download_era5_year(2021)

In [ ]:
# ERA5 2022 ────────────────────────────────────────────────────────
download_era5_year(2022)

In [ ]:
# Merge all yearly CSVs → era5_hourly_2017_2022_bst.csv

import pandas as pd, os, glob

CSV_DIR  = "/kaggle/input/datasets/aditybarua07/era5-data"
OUT_PATH = "/kaggle/working/output/csv/era5_hourly_2017_2022_bst.csv"

csv_files = sorted(glob.glob(f"{CSV_DIR}/era5_20??.csv"))
print(f"Found {len(csv_files)} yearly CSVs:")
for f in csv_files:
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.2f} MB)")

missing = [y for y in range(2017, 2023)
           if f"{CSV_DIR}/era5_{y}.csv" not in csv_files]
if missing:
    print(f"\n⚠️  Missing years: {missing} — run their cells first.")
else:
    dfs = []
    for f in csv_files:
        df = pd.read_csv(f, parse_dates=["timestamp_utc", "timestamp_bst"])
        print(f"  {os.path.basename(f)}: {len(df):,} rows  "
              f"{df['timestamp_bst'].iloc[0]} → {df['timestamp_bst'].iloc[-1]}")
        dfs.append(df)

    merged = (pd.concat(dfs, ignore_index=True)
                .sort_values("timestamp_bst")
                .reset_index(drop=True))

    before = len(merged)
    merged = merged.drop_duplicates(subset="timestamp_bst")
    if len(merged) < before:
        print(f"  ⚠️  Dropped {before - len(merged)} duplicate BST timestamps")

    merged.to_csv(OUT_PATH, index=False)
    print(f"\n✅ Saved: era5_hourly_2017_2022_bst.csv")
    print(f"   Rows   : {len(merged):,}")
    print(f"   Cols   : {list(merged.columns)}")
    print(f"   Period : {merged['timestamp_bst'].min()} → "
          f"{merged['timestamp_bst'].max()}")
    print(f"   Size   : {os.path.getsize(OUT_PATH)/1e6:.2f} MB")